# Backfill Historical S&P 500 Constituents

Fills in every missing `sp500_constituents_YYYYMM.csv` file using the historical
[fja05680/sp500](https://github.com/fja05680/sp500) dataset for constituent membership
and Yahoo Finance for price history.

**Market-cap weight estimation**  
Historical shares outstanding is not freely available, so weights are approximated as:

```
hist_mcap(sym, month) ≈ current_mcap(sym) × (hist_price / current_price)
```

This is accurate enough to identify rank-1 (largest market cap) each month, which
has historically been one of a small set of companies (GE → XOM → AAPL → MSFT ↔ NVDA).

In [1]:
import re
import warnings
from datetime import date
from io import StringIO
from pathlib import Path

import pandas as pd
import requests
import yfinance as yf

warnings.filterwarnings("ignore")

CONSTITUENTS_DIR = Path("sp500_constituents")
CONSTITUENTS_DIR.mkdir(parents=True, exist_ok=True)

# ── configurable start month (YYYYMM) ──────────────────────────────────────
START_MONTH = "200001"   # January 2000
# ───────────────────────────────────────────────────────────────────────────

GITHUB_RAW = "https://raw.githubusercontent.com/fja05680/sp500/master/"

## 1 · Find missing months

In [2]:
existing = {
    p.stem.split("_")[-1]
    for p in CONSTITUENTS_DIR.glob("sp500_constituents_*.csv")
}

today = date.today()
all_months = pd.period_range(
    start=START_MONTH,
    end=today.strftime("%Y%m"),
    freq="M",
)
all_months_str = [p.strftime("%Y%m") for p in all_months]
missing = sorted(set(all_months_str) - existing)

print(f"Existing files : {sorted(existing)}")
print(f"Missing months : {len(missing)}  ({missing[0]} → {missing[-1]})")

Existing files : ['202603']
Missing months : 314  (200001 → 202602)


## 2 · Download historical constituent data from GitHub

In [3]:
# The filename contains the last update date in (MM-DD-YYYY) format.
# We try the latest known version; fall back to the unversioned file.
HIST_FILE = "S%26P%20500%20Historical%20Components%20%26%20Changes%2801-17-2026%29.csv"

url = GITHUB_RAW + HIST_FILE
resp = requests.get(url, timeout=60)
resp.raise_for_status()
print(f"Downloaded {len(resp.content):,} bytes from GitHub")

hist_raw = pd.read_csv(StringIO(resp.text), index_col=0, header=0)
hist_raw.index = pd.to_datetime(hist_raw.index)
hist_raw.sort_index(inplace=True)

print(f"Shape: {hist_raw.shape}")
print(f"Date range: {hist_raw.index[0].date()} → {hist_raw.index[-1].date()}")
print("\nFirst 2 rows (head):")
hist_raw.head(2)

Downloaded 5,499,082 bytes from GitHub
Shape: (2705, 1)
Date range: 1996-01-02 → 2026-01-14

First 2 rows (head):


,tickers
date,
1996-01-02,"AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD..."
1996-01-03,"AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD..."


In [4]:
# The GitHub file has a single "tickers" column whose values are
# comma-separated ticker strings, e.g. "AAPL,MSFT,AMZN,..."
print(f"Columns : {hist_raw.columns.tolist()}")
print(f"Sample  : {str(hist_raw.iloc[0, 0])[:80]} …")


def get_symbols_for_date(target: pd.Timestamp) -> set:
    """Return the S&P 500 symbol set on the last available date ≤ target."""
    idx = hist_raw.index.get_indexer([target], method="pad")[0]
    if idx == -1:
        return set()
    tickers_str = hist_raw.iloc[idx, 0]
    if not isinstance(tickers_str, str):
        return set()
    return {t.strip() for t in tickers_str.split(",") if t.strip()}


# Sanity-check
sample = get_symbols_for_date(pd.Timestamp("2020-01-01"))
print(f"\nS&P 500 members on 2020-01-01 : {len(sample)} symbols")
print("Sample (sorted):", sorted(sample)[:10])

Columns : ['tickers']
Sample  : AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,ADP,ADSK,AEE,AEP,AET,AGC,AGN,AHM,AIG …

S&P 500 members on 2020-01-01 : 505 symbols
Sample (sorted): ['A', 'AAL', 'AAP', 'AAPL', 'ABBV', 'ABC', 'ABMD', 'ABT', 'ACN', 'ADBE']


## 3 · Build company-name lookup and current market caps

In [5]:
# Parse every row of the tickers column to collect all unique historical symbols
all_hist_symbols: set[str] = set()
for tickers_str in hist_raw.iloc[:, 0].dropna():
    all_hist_symbols.update(t.strip() for t in tickers_str.split(",") if t.strip())

# Also add symbols from any existing monthly constituent CSVs
for csv_f in CONSTITUENTS_DIR.glob("sp500_constituents_*.csv"):
    _df = pd.read_csv(csv_f, usecols=["symbol"])
    all_hist_symbols.update(_df["symbol"].tolist())

all_hist_symbols = sorted(all_hist_symbols)
print(f"Total unique symbols across history: {len(all_hist_symbols)}")

Total unique symbols across history: 1194


In [6]:
# Market caps come entirely from yfinance (existing CSVs only contain symbol now)
mcap_lookup: dict[str, float] = {}
print(f"Will fetch market caps for {len(all_hist_symbols)} symbols from yfinance …")

Will fetch market caps for 1194 symbols from yfinance …


In [7]:
# ── fetch current market caps from yfinance for remaining symbols ────────────
# This runs once; only queries symbols not already seeded.
need_mcap = [s for s in all_hist_symbols if s not in mcap_lookup]
print(f"Fetching market cap for {len(need_mcap)} additional symbols via yfinance …")

for sym in need_mcap:
    try:
        fi = yf.Ticker(sym).fast_info
        mc = getattr(fi, "market_cap", None)
        if mc and mc > 0:
            mcap_lookup[sym] = mc
        lname = getattr(fi, "currency", None)  # fast_info has no name
    except Exception:
        pass

print(f"Market cap data available for {len(mcap_lookup)} symbols")

Fetching market cap for 1194 additional symbols via yfinance …


$AABA: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")
$AAMRQ: possibly delisted; no price data found  (period=5d)
$ABC: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ABKFQ"}}}
$ABKFQ: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")
$ABMD: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")
$ABS: possibly delisted; no price data found  (period=5d)
$ACAS: possibly delisted; no price data found  (period=5d)
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ACKH"}}}
$ACKH: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be deli

Market cap data available for 722 symbols


## 4 · Download historical price data (one bulk yfinance call)

In [8]:
# Only download symbols for which we have a market cap reference
dl_symbols = sorted(mcap_lookup.keys())

dl_start = (pd.to_datetime(START_MONTH, format="%Y%m") - pd.DateOffset(days=10)).strftime("%Y-%m-%d")
dl_end   = (pd.to_datetime(today.strftime("%Y%m"), format="%Y%m") + pd.DateOffset(months=1)).strftime("%Y-%m-%d")

print(f"Downloading {len(dl_symbols)} symbols from {dl_start} to {dl_end} …")
print("(This may take a few minutes for the first run)")

raw_prices = yf.download(
    dl_symbols,
    start=dl_start,
    end=dl_end,
    auto_adjust=True,
    progress=True,
    group_by="ticker",
)

# Flatten to a (date × symbol) Close price DataFrame
if isinstance(raw_prices.columns, pd.MultiIndex):
    prices = raw_prices.xs("Close", axis=1, level=1)
else:
    prices = raw_prices[["Close"]].rename(columns={"Close": dl_symbols[0]})

prices.index = pd.to_datetime(prices.index)
prices.sort_index(inplace=True)
print(f"\nPrice matrix: {prices.shape}")

(This may take a few minutes for the first run)


[*********************100%***********************]  722 of 722 completed



Price matrix: (6586, 722)


## 5 · Compute estimated market caps and write missing CSVs

In [9]:
# Last available closing price per symbol (used as the "current price" reference)
current_price = prices.iloc[-1].dropna()


def rank_by_mcap(symbols: set, month_prices_row: pd.Series) -> list[str]:
    """
    Estimate market cap for each symbol in *symbols* for a given month as:
        hist_mcap ≈ current_mcap(sym) × (hist_price / current_price)
    Returns symbols sorted descending by estimated market cap.
    Symbols with no market cap or price data are excluded.
    """
    mcaps: dict[str, float] = {}
    for sym in symbols:
        if sym not in mcap_lookup:
            continue
        hist_p = month_prices_row.get(sym, float("nan"))
        cur_p  = current_price.get(sym, float("nan"))
        if pd.isna(hist_p) or pd.isna(cur_p) or cur_p == 0:
            continue
        mcaps[sym] = mcap_lookup[sym] * (hist_p / cur_p)
    return [sym for sym, _ in sorted(mcaps.items(), key=lambda x: x[1], reverse=True)]


saved, skipped = 0, 0

for month_str in missing:
    target_dt = pd.to_datetime(month_str, format="%Y%m")

    # ── constituent symbols for this month ───────────────────────────────
    symbols = get_symbols_for_date(target_dt)
    if not symbols:
        print(f"  {month_str} → no constituent data, skipped")
        skipped += 1
        continue

    # ── first trading-day prices for this specific year-month ────────────
    month_end = target_dt + pd.offsets.MonthEnd(1)
    month_slice = prices.loc[target_dt : month_end]
    if month_slice.empty:
        print(f"  {month_str} → no price data, skipped")
        skipped += 1
        continue
    month_prices_row = month_slice.iloc[0]   # first trading day of that month

    # ── rank by estimated market cap ─────────────────────────────────────
    ranked_symbols = rank_by_mcap(symbols, month_prices_row)
    if not ranked_symbols:
        print(f"  {month_str} → insufficient market cap data, skipped")
        skipped += 1
        continue

    # ── save: one symbol per row, row 0 = rank 1 ─────────────────────────
    df_out = pd.DataFrame({"symbol": ranked_symbols})
    csv_path = CONSTITUENTS_DIR / f"sp500_constituents_{month_str}.csv"
    df_out.to_csv(csv_path, index=False)
    saved += 1

print(f"\nDone — saved {saved} files, skipped {skipped} months")


Done — saved 314 files, skipped 0 months


## 6 · Verify: rank-1 stock per year

In [10]:
records = []
for csv_f in sorted(CONSTITUENTS_DIR.glob("sp500_constituents_*.csv")):
    month_tag = csv_f.stem.split("_")[-1]
    if len(month_tag) != 6:
        continue
    try:
        df = pd.read_csv(csv_f, usecols=["symbol"])
        records.append({
            "month" : pd.to_datetime(month_tag, format="%Y%m"),
            "rank_1": df["symbol"].iloc[0],   # row 0 = rank 1 (highest market cap)
        })
    except Exception:
        pass

all_holdings = pd.DataFrame(records).sort_values("month").reset_index(drop=True)
print(f"Total months with data: {len(all_holdings)}")

# Show January of each year to verify rank-1 changes over time
yearly = all_holdings[all_holdings["month"].dt.month == 1]
display(yearly.reset_index(drop=True))

Total months with data: 315


,month,rank_1
0,2000-01-01,AIG
1,2001-01-01,AIG
2,2002-01-01,AIG
3,2003-01-01,AIG
4,2004-01-01,C
5,2005-01-01,C
6,2006-01-01,C
7,2007-01-01,C
8,2008-01-01,AIG
9,2009-01-01,XOM
